## This notebook covers basic Databricks Transformations in both PySpark & SQL

reading the table and storing it in a dataframe and displaying it

In [0]:
df=spark.read.table("samples.bakehouse.sales_franchises")
df.display()

doing the same thing in sql now using **%sql magic command**

In [0]:
%sql
select * from samples.bakehouse.sales_franchises

Get the columns - PySpark 

In [0]:
df.columns

Get all the table columns - SQL

In [0]:
%sql
show columns from samples.bakehouse.sales_franchises

Get specific table columns using Select - PySpark

In [0]:
df2=df.select("name","franchiseID","country","city","zipcode")
df2.display()

Get specific columns - SQL

In [0]:
%sql
select name,franchiseId,country,city,zipCode from samples.bakehouse.sales_franchises

Using expressions in PySpark (Expression is basically a function in the pyspark.sql.function library which we import.) Everything inside the expr() should be in "".

In [0]:
from pyspark.sql.functions import expr
# df2=df.select("name",expr("franchiseID as id"),expr("country as c"),expr("city as ct"),expr("zipcode as z"))
df2=df.select("name", expr("concat(city,' - ',country) as location"))
df2.display()                               

SelectExpr - PySpark. Basically same concept as Expr but everything inside SelectExpr() is treated as an expression

In [0]:
from pyspark.sql.functions import expr
df2=df.selectExpr("name","concat(city,' - ',country) as location")
df2.display()

Alias - PySpark

In [0]:
from pyspark.sql.functions import expr
df2=df.select("name",expr("concat(city,'_',country)").alias("location"))
display(df2)

Alias - SQL

In [0]:
%sql
select name, concat(city, ' - ', country) as location from samples.bakehouse.sales_franchises

withColumn in PySpark - used to add a new column to our existing table

In [0]:
from pyspark.sql.functions import expr
df2=df.withColumn("location",expr("concat(city,' - ',country)"))
df2.display()

Adding a new column in SQL

In [0]:
%sql
select *, concat(city, ' - ', country) as location from samples.bakehouse.sales_franchises

Dropping a column - PySpark

In [0]:
df2=df.drop("latitude","longitude")
df2.display()

Dropping a column - SQL

In [0]:
%sql
select * except(longitude,latitude) from samples.bakehouse.sales_franchises

Data Type Casting - PySpark (Changing data type of cols)

In [0]:
from pyspark.sql.functions import col
df2=df.withColumn("name_varchar", col("name").cast("varchar(255)"))
#if i want to change data type of existing column just use its original name
# df2=df.withColumn("name", col("name").cast("varchar(255)"))
df2.display()

Data casting - SQL

In [0]:
%sql
select *,cast(supplierId as double) as supplier_id_double from samples.bakehouse.sales_franchises

Filtering records - PySpark

In [0]:
from pyspark.sql.functions import col
df2=df.where(col("country")=="US")
df2.display()

Filtering method 2 - using **.filter()**

In [0]:
df2=df.filter(col("country")=="US")
df2.display()

Filtering - SQL

In [0]:
%sql
select * from samples.bakehouse.sales_franchises where country='US'

Selecting DISTINCT records - PySpark

In [0]:
#select all rows that are distinct
# df2=df.distinct()
#select records based on selective columns that are distinct
df2=df.select("country","city").distinct()
df2.display()

Select DISTINCT records - SQL

In [0]:
%sql
select distinct country,city from samples.bakehouse.sales_franchises

Union operation - PySpark

In [0]:
df2=df
df3=df.union(df2)
df3.display()


Union All - PySpark (Union in SQL removes dups but Union All does not remove dups). In PySpark **.union()** & **.unionAll()** does not remove dups

In [0]:
df3=df.unionAll(df2)
df3.display()

SQL - Union Operation

In [0]:
%sql
select * from samples.bakehouse.sales_franchises 
union
select * from samples.bakehouse.sales_franchises

SQL - Union All operation

In [0]:
%sql
select * from samples.bakehouse.sales_franchises
union all
select * from samples.bakehouse.sales_franchises

OrderBy function - PySpark

In [0]:
from pyspark.sql.functions import desc, col
df2=df.orderBy(col('country'))
# df2=df.orderBy(col('country').desc())
df2.display()


Order by - SQL

In [0]:
%sql
select * from samples.bakehouse.sales_franchises order by country desc

Limit - PySpark

In [0]:
df2=df.limit(10)
df2.display()

Limit - SQL

In [0]:
%sql
Select * from samples.bakehouse.sales_franchises limit(10)

### Repartition & Coalesce 
This basically affects the number of partitions we have for parallel processing purposes. 
1. Repartition

Repartitioning changes the number of partitions by performing a **full shuffle**, where data is redistributed across partitions.

- It can increase or decrease the number of partitions.
- It redistributes data across partitions, which can help improve parallelism and balance data distribution.
- Since it involves shuffling data across the network and potentially disk I/O, it is relatively expensive.
- It creates a **shuffle dependency**.

**Example:**

```python
df2 = df.repartition(8)
print(df2.rdd.getNumPartitions())
```
2. Coalesce

Coalesce is primarily used to **reduce the number of partitions without performing a full shuffle**.

- It is generally more efficient than `repartition()` when reducing partitions.
- It groups multiple input partitions into fewer output partitions.
- It does not necessarily redistribute data evenly, so output partitions may have uneven amounts of data.
- It typically creates a **narrow dependency**.

**Example:**

```python
df2 = df.coalesce(2)
print(df2.rdd.getNumPartitions())

In [0]:
df2=df.repartition(2)
df2=df.coalesce(2)
df2.display()

Command to get the number of partitions
**df.rdd.getNumberPartitons()** \
However, for free serverless compute this does not work

In [0]:
df.rdd.getNumberPartitions()

For collecting df values into an array (Python object for further manipulation) we use the following function **.collect()**

In [0]:
df.collect()

converting a python df to a sql view to read and query from that view we use **.createOrReplaceTempView()**

In [0]:
df.limit(5).createOrReplaceTempView("temp")


sql code to interact with the temp view

In [0]:
%sql
select * from temp

creating a df out of a view in sql

In [0]:
df2=spark.sql("select * from temp limit 2")
df2.display()